# TabS61 – Accuracy, ARI & Avg Silhouette

Reads `.Rds` simulation outputs from the **enhanced** pipeline across all
scenarios and produces the LaTeX summary table
(Accuracy · ARI · Avg Silhouette, by scenario / γ, administrative censoring,
k = 20, C = 3).

**Expected directory layout** (mirrors `enhanced_simulation_main.R`):
```
output/
  tabS61/
    baseline/
      sim_seed*_c*_k*_gammapar*_frailty*_censor*.Rds
    imbalanced_moderate/
      ...
    (etc.)
```

In [1]:
# ── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import rdata
from pathlib import Path
from sklearn.metrics import adjusted_rand_score

# ── Configuration ────────────────────────────────────────────────────────────
BASE_DIR  = Path("output/tabS61")          # root of saved .Rds files
N_CLUSTERS_EXPECTED = 3             # keep only runs where n_components == 3

# All scenarios to process (order matches the LaTeX table rows a–i)
SCENARIOS = [
    "baseline",
    "imbalanced_moderate",
    "imbalanced_severe",
    "medium_separation",
    "weak_separation",
    "misspec_frailty",
    "misspec_baseline",
    "misspec_both",
    "worst_case",
]

# Human-readable labels for the LaTeX table (rows a–i)
SCENARIO_LABELS = {
    "baseline":            r"(a) Baseline",
    "imbalanced_moderate": r"(b) Moderate imb",
    "imbalanced_severe":   r"(c) Severe imb",
    "medium_separation":   r"(d) Medium sep",
    "weak_separation":     r"(e) Low sep",
    "misspec_frailty":     r"(f) Frailty missp",
    "misspec_baseline":    r"(g) Baseline missp",
    "misspec_both":        r"(h) Joint missp",
    "worst_case":          r"(i) Worst-case",
}


In [2]:
# ── Helper: greedy relabelling (predicted → true) ─────────────────────────────
def simple_relabel(true_labels, pred_labels):
    """Map predicted cluster ids to true ids by maximum overlap (greedy)."""
    true_labels = pd.Series(true_labels)
    pred_labels = pd.Series(pred_labels)

    mapping = {}
    available_true = set(true_labels.unique())

    for p in pred_labels.unique():
        overlaps = {
            t: ((true_labels == t) & (pred_labels == p)).sum()
            for t in available_true
        }
        best = max(overlaps, key=overlaps.get)
        mapping[p] = best
        available_true.discard(best)

    return pred_labels.map(mapping)

In [3]:
# ── Load & process all .Rds files ─────────────────────────────────────────────
#
# Each file is an R list saved with saveRDS().
#
# Expected fields (from enhanced_simulation_main.R):
#   seed, scenario, c, k, gammapar, n_components, clusters (Nx2 matrix),
#   dgp_frailty, dgp_baseline, fit_frailty, fit_baseline,
#   separation, balance, censoring_rate, silhouette
#
# The censoring type is encoded in the filename:
#   _censoradministrative_  or  _censornormal_

results = {}   # (scenario, gamma) -> {"acc": [], "ari": [], "sil": [], "n": int}
skipped = 0

for scenario in SCENARIOS: # need to do this because there is an extra layer
    folder = BASE_DIR / scenario 
    if not folder.exists():
        print(f"[WARNING] Folder not found: {folder}")
        continue

    rds_files = sorted(folder.glob("*.Rds"))

    for file in rds_files:
        fname = file.name

        # ── censoring filter from filename ──────────────
        # here we are only dealing with administrative cenrosing - see Tab2 to change
        if "censoradministrative" not in fname:
            skipped += 1
            continue

        if "frailty_intensity0.5" in fname:
            frailty_intensity = 0.5
        elif "frailty_intensity1.5" in fname:
            frailty_intensity = 1.5
        elif "frailty_intensity1" in fname:
            frailty_intensity = 1
        else:
            frailty_intensity = None
        
         # ── Read the Rds file ──────────────────────────
        try:
            obj = rdata.read_rds(file)
            obj = {str(k): v for k, v in obj.items()}
        except Exception as e:
            print(f"  [WARNING] Could not read {fname}: {e}")
            skipped += 1
            continue

        # ── Filter: only runs with the expected number of components ───────
        n_comp = obj.get("n_components", None)
        if n_comp is None or int(np.asarray(n_comp).item()) != N_CLUSTERS_EXPECTED:
            skipped += 1
            continue

        # ── Extract gamma ───────────────────────────────────────
        # here we are fixing k, so we are not taking it out - see Tab2 to change
        gamma = float(np.asarray(obj["gammapar"]).item())

        # ── Compute accuracy & ARI ────────────────────────────────────
        clusters_arr = np.asarray(obj["clusters"])

        if clusters_arr.ndim != 2 or clusters_arr.shape[1] != 2:
            print(f"[WARNING] Bad clusters shape in {fname}: {clusters_arr.shape}")
            skipped += 1
            continue

        true_labels = clusters_arr[:, 0]
        pred_labels = clusters_arr[:, 1]

        # relabel + numpy arrays
        aligned = simple_relabel(true_labels, pred_labels)

        true_labels = np.asarray(true_labels)
        aligned     = np.asarray(aligned)

        accuracy = (true_labels == aligned).mean()
        ari      = adjusted_rand_score(true_labels, aligned)

        # ── silhouette ──────────────────────────────────
        sil_raw = obj.get("silhouette", None)
        avg_sil = np.nan

        if isinstance(sil_raw, dict) and "si.summary.Mean" in sil_raw:
            avg_sil = float(np.asarray(sil_raw["si.summary.Mean"]).item())

        # ── KEY FIX: include scenario + gamma ───────────
        key = (scenario, gamma, frailty_intensity)

        if key not in results:
            results[key] = {
                "acc": [],
                "ari": [],
                "sil": []
            }

        results[key]["acc"].append(accuracy)
        results[key]["ari"].append(ari)
        results[key]["sil"].append(avg_sil)

# ── add n explicitly for LaTeX table ─────────────────
for k in results:
    results[k]["n"] = len(results[k]["acc"])

total_loaded = sum(results[k]["n"] for k in results)
print(f"Processed {total_loaded} files | skipped {skipped}")

Processed 5393 files | skipped 7


In [4]:
# ── Build summary DataFrame ───────────────────────────────────────────────────
rows = []
for (scenario, gamma, frailty_intensity), vals in results.items():
    acc = np.array(vals["acc"])
    ari = np.array(vals["ari"])
    sil = np.array(vals["sil"])

    rows.append({
        "scenario"   : scenario,
        "gamma"      : gamma,
        "frailty_intensity": frailty_intensity,
        "n"          : len(acc),
        "Acc_mean"   : np.round(acc.mean(),        3),
        "Acc_median" : np.round(np.median(acc),    3),
        "Acc_sd"     : np.round(acc.std(),         3),
        "ARI_mean"   : np.round(ari.mean(),        3),
        "ARI_median" : np.round(np.median(ari),    3),
        "ARI_sd"     : np.round(ari.std(),         3),
        "Sil_mean"   : np.round(np.nanmean(sil),   3),
        "Sil_median" : np.round(np.nanmedian(sil), 3),
        "Sil_sd"     : np.round(np.nanstd(sil),    3),
    })

# ── Sort: follow the SCENARIOS order, then gamma ascending ────────────────
scenario_order = {s: i for i, s in enumerate(SCENARIOS)}
summary_df = (
    pd.DataFrame(rows)
    .assign(_sord=lambda df: df["scenario"].map(scenario_order))
    .sort_values(["_sord", "gamma", "frailty_intensity"])
    .drop(columns="_sord")
    .reset_index(drop=True)
)

print(summary_df.to_string())

               scenario   gamma  frailty_intensity    n  Acc_mean  Acc_median  Acc_sd  ARI_mean  ARI_median  ARI_sd  Sil_mean  Sil_median  Sil_sd
0              baseline  0.0001                0.5  100     0.971       1.000   0.089     0.942       1.000   0.161     0.792       0.832   0.116
1              baseline  0.0001                1.0  100     0.977       1.000   0.061     0.949       1.000   0.127     0.792       0.831   0.099
2              baseline  0.0001                1.5  100     0.973       1.000   0.074     0.946       1.000   0.136     0.788       0.831   0.112
3              baseline  0.1000                0.5  100     0.950       1.000   0.083     0.888       1.000   0.179     0.739       0.823   0.147
4              baseline  0.1000                1.0  100     0.957       1.000   0.074     0.904       1.000   0.156     0.752       0.820   0.122
5              baseline  0.1000                1.5  100     0.949       1.000   0.082     0.888       1.000   0.167     0.74

# LaTex table

In [144]:
# ── LaTeX table generator ─────────────────────────────────────────────────────
#
# Produces the 11-column table matching the paper format:
#   Scenario | gamma | n | Acc (Mean/Median/SD) | ARI (Mean/Median/SD)
#                        | Avg Silhouette (Mean/Median/SD)

def fmt_gamma(g):
    """Format a gamma value as LaTeX scientific / decimal notation."""
    if g == 0:
        return "$0$"
    elif g < 0.01:
        e = int(round(np.log10(g)))
        return f"$10^{{{e}}}$"
    else:
        return f"${g:g}$"


def fmt_val(v):
    """Format a metric value to 3 decimal places."""
    return f"{v:.3f}"


def build_latex_table(df, scenario_labels, scenarios_order):
    """
    Build the LaTeX table string from summary_df.

    Parameters
    ----------
    df               : summary DataFrame (one row per scenario x gamma)
    scenario_labels  : dict mapping scenario key -> LaTeX display string
    scenarios_order  : list of scenario keys in desired display order
    """
    lines = []

    # Header
    lines.append(r"\begin{table}[!htbp]")
    lines.append(r"\centering")
    lines.append(r"\footnotesize")
    lines.append(r"\begin{tabular}{llll ccc ccc ccc}")
    lines.append(r"\toprule")
    lines.append(r"\textbf{Scenario} & $\gamma$ & $\theta$ & $n$")
    lines.append(r"    & \multicolumn{3}{c}{\textbf{Accuracy}}")
    lines.append(r"    & \multicolumn{3}{c}{\textbf{ARI}} & \multicolumn{3}{c}{\textbf{Avg Silhouette}} \\")
    lines.append(r"\cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}")
    lines.append(r" & & & & Mean & Median & SD & Mean & Median & SD & Mean & Median & SD \\")
    lines.append(r"\toprule")

    # Data rows
    for si, scenario in enumerate(scenarios_order):
        sdf = df[df["scenario"] == scenario].sort_values("gamma")
        if sdf.empty:
            continue

        label    = scenario_labels.get(scenario, scenario)
        n_rows_s = len(sdf)

        if si > 0:
            lines.append(r"\cmidrule(lr){1-13}")

        for ri, (_, row) in enumerate(sdf.iterrows()):
            # Column 1: scenario label (multirow, first row of block only)
            if ri == 0:
                col1 = f"\\multirow{{{n_rows_s}}}{{*}}{{{label}}}"
            else:
                col1 = ""

            gamma_str = fmt_gamma(row["gamma"])
            n_val     = int(row["n"])

            data_cols = " & ".join([
                fmt_val(row["Acc_mean"]),
                fmt_val(row["Acc_median"]),
                fmt_val(row["Acc_sd"]),
                fmt_val(row["ARI_mean"]),
                fmt_val(row["ARI_median"]),
                fmt_val(row["ARI_sd"]),
                fmt_val(row["Sil_mean"]),
                fmt_val(row["Sil_median"]),
                fmt_val(row["Sil_sd"]),
            ])

            lines.append(f"  {col1} & {gamma_str} & {row['frailty_intensity']} & {n_val} & {data_cols}\\\\")

    # Footer
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(
        r"\caption{\small Empirical means, medians, and standard deviations of "
        r"\textit{accuracy}, \textit{ARI}, and \textit{average silhouette} across "
        r"$B=100$ simulation replicates, evaluated for $k=20$, administrative censoring "
        r"and $\gamma \in\{0.001, 0.1\}$, with $C=3$ true clusters. The column $n$ "
        r"denotes the number of runs (out of 100) where the selected number of clusters "
        r"equals the true value ($C=3$); performance metrics are computed over these "
        r"runs only.}"
    )
    lines.append(r"\label{tab:newsim_AccuracyARISil}")
    lines.append(r"\end{table}")

    return "\n".join(lines)


latex_str = build_latex_table(summary_df, SCENARIO_LABELS, SCENARIOS)
print(latex_str)

\begin{table}[!htbp]
\centering
\footnotesize
\begin{tabular}{llll ccc ccc ccc}
\toprule
\textbf{Scenario} & $\gamma$ & $\theta$ & $n$
    & \multicolumn{3}{c}{\textbf{Accuracy}}
    & \multicolumn{3}{c}{\textbf{ARI}} & \multicolumn{3}{c}{\textbf{Avg Silhouette}} \\
\cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}
 & & & & Mean & Median & SD & Mean & Median & SD & Mean & Median & SD \\
\toprule
  \multirow{6}{*}{(a) Baseline} & $10^{-4}$ & 0.5 & 100 & 0.971 & 1.000 & 0.089 & 0.942 & 1.000 & 0.161 & 0.792 & 0.832 & 0.116\\
   & $10^{-4}$ & 1.0 & 100 & 0.977 & 1.000 & 0.061 & 0.949 & 1.000 & 0.127 & 0.792 & 0.831 & 0.099\\
   & $10^{-4}$ & 1.5 & 100 & 0.973 & 1.000 & 0.074 & 0.946 & 1.000 & 0.136 & 0.788 & 0.831 & 0.112\\
   & $0.1$ & 0.5 & 100 & 0.950 & 1.000 & 0.083 & 0.888 & 1.000 & 0.179 & 0.739 & 0.823 & 0.147\\
   & $0.1$ & 1.0 & 100 & 0.957 & 1.000 & 0.074 & 0.904 & 1.000 & 0.156 & 0.752 & 0.820 & 0.122\\
   & $0.1$ & 1.5 & 100 & 0.949 & 1.000 & 0.082 & 0.888 & 1.000 & 